# Databricks/PySpark for Data Analysts/Engineers — Module 10: Delta Lake -- Basics

This module is different from previous ones: through Modules 1-9, every piece of code
could run locally. **Delta Lake** requires downloading additional Java libraries from
the internet the first time it's used (or native support, as in Databricks) -- in some
local environments (including the one this course was prepared in), downloading these
libraries is blocked by network policy. The code in this module is correct and ready to
run in **Databricks Free Edition** (where Delta Lake is built in natively, with no
configuration needed) -- we mark this clearly at each section.

> ⚠️ **This module was not run on the same execution engine as Modules 1-9**
>
> Unlike earlier modules, the code in this notebook was NOT automatically verified by running it in the same environment this course was prepared in (access to the Maven repository, where Spark downloads the Delta Lake library, is blocked there). The syntax exactly matches the official Delta Lake documentation -- but if you run this notebook on YOUR OWN computer with full internet access, the first run may take longer (Spark will download the needed `.jar` files). In Databricks Free Edition, everything works right away, with no downloading at all.

## Table of contents
1. [Why Delta Lake -- the problems with plain Parquet](#sec1)
2. [ACID transactions and _delta_log](#sec2)
3. [Time travel: traveling through data history](#sec3)
4. [MERGE INTO -- an upsert in one statement](#sec4)
5. [Schema evolution](#sec5)
6. [Summary and exercises](#sec6)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Why Delta Lake -- the problems with plain Parquet

In Module 6 we wrote data to plain Parquet. It's a good format, but it has serious
limitations for team work and production data pipelines:

- **No ACID transactions** -- if a Parquet write is interrupted halfway through (a
crash, a cluster restart), the folder can be left in an INCONSISTENT state -- some
files written, some not.
- **No schema enforcement** -- nothing stops someone from writing data with a
DIFFERENT schema (e.g. a different column type) into the same folder, breaking every
future read.
- **No history** -- once you overwrite data (`mode("overwrite")`), the OLD version is
gone for good. There's no way to "roll back" to the state before a bad write.
- **No real UPDATE/DELETE** -- Parquet is a format ONLY for reading/writing whole files
-- there's no notion of "change this one row", you have to rewrite the whole file.

**Delta Lake** is a layer ON TOP OF Parquet (the data files are still plain
`.parquet` files) that adds a transaction log solving all of the problems above.

> ⚡ **Delta Lake is not a different file format -- it's Parquet + a transaction log**
>
> The key thing to understand: data in a Delta table is physically stored as plain Parquet files. What Delta Lake adds is a `_delta_log/` folder alongside them, containing the history of every operation. That's why Delta Lake stays backward compatible with tools that read Parquet, while adding database-like functionality on top.

<a id="sec2"></a>
## 2. ACID transactions and _delta_log

Writing in the `delta` format (instead of `parquet`) creates a `_delta_log/` subfolder
in the target folder -- a sequence of numbered JSON files
(`00000000000000000000.json`, `00000000000000000001.json`, ...), each describing ONE
transaction: which Parquet files were added, which were removed, and what the schema
was at that moment.

In [ ]:
# Writing in delta format -- almost identical syntax to Module 6, just a different format
customers.write.format("delta").mode("overwrite").save("delta_data/customers_delta")

# Reading -- also almost identical to parquet
customers_delta = spark.read.format("delta").load("delta_data/customers_delta")
customers_delta.show(5)


In [ ]:
# DeltaTable -- a special object for Delta-specific operations (history, MERGE, ...)
from delta.tables import DeltaTable

customers_table = DeltaTable.forPath(spark, "delta_data/customers_delta")
customers_table.toDF().show(5)


> ⚠️ **Every write is a NEW transaction, not an in-place overwrite**
>
> Even `mode("overwrite")` in Delta Lake does NOT physically delete the old files right away -- it appends a new entry to `_delta_log/`, saying 'the old files are now inactive, use these new ones instead'. That's exactly what enables time travel in section 3 -- the old Parquet files are still physically sitting on disk until you run the `VACUUM` command (which cleans up old, unused files past the retention period).

<a id="sec3"></a>
## 3. Time travel: traveling through data history

Because every change to a Delta table is a separate, numbered entry in the log, you can
read ANY earlier version of the table -- by version number or by timestamp.

In [ ]:
# The full history of every change to the table
customers_table.history().select("version", "timestamp", "operation").show(truncate=False)


In [ ]:
# Simulate a second change to the table, so we have something to look at in the history
half_of_customers = customers.filter(customers.customer_id <= 150)
half_of_customers.write.format("delta").mode("overwrite").save("delta_data/customers_delta")

# Version 0 (before the overwrite) -- read via the versionAsOf option
version_0 = spark.read.format("delta").option("versionAsOf", 0).load("delta_data/customers_delta")
print(f"Version 0: {version_0.count()} rows")

# The latest version
latest = spark.read.format("delta").load("delta_data/customers_delta")
print(f"Latest version: {latest.count()} rows")


In [ ]:
# The same time travel in Spark SQL (SELECT ... VERSION AS OF / TIMESTAMP AS OF)
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM delta.`delta_data/customers_delta` VERSION AS OF 0
""").show()


> 🧱 **RESTORE -- reverting a table to an older version FOR GOOD**
>
> Reading via `VERSION AS OF`/`versionAsOf` only PEEKS at an old version -- it doesn't change the table's current state. If you want to truly REVERT a table to an earlier version (e.g. after an accidental bad write), that's what `RESTORE TABLE ... TO VERSION AS OF ...` is for -- this too creates a new entry in the history, so the `RESTORE` itself can also be undone later.

<a id="sec4"></a>
## 4. MERGE INTO -- an upsert in one statement

`MERGE INTO` is the most important Delta Lake operation for data pipelines: in ONE
statement it joins a target table with a source of new/changed data and decides, for
each row, whether to **update** it (if it already exists), **insert** it (if it's
new), or (optionally) **delete** it. This is exactly the operation known as an
*upsert*.

In [ ]:
# New data: updating the city for some customers + brand new customers
from pyspark.sql import Row

new_customer_data = spark.createDataFrame([
    Row(customer_id=1, city="Wroclaw", segment="Premium", days_since_signup=500),
    Row(customer_id=301, city="Poznan", segment="New", days_since_signup=1),
    Row(customer_id=302, city="Gdansk", segment="New", days_since_signup=1),
])

customers_table.alias("target").merge(
    new_customer_data.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(set={
    "city": "source.city",
    "segment": "source.segment",
}).whenNotMatchedInsertAll().execute()

customers_table.toDF().orderBy("customer_id").show(5)


> ⚡ **The same MERGE INTO in Spark SQL**
>
> The exact same operation in plain SQL looks like this:

```sql
MERGE INTO customers_delta AS target
USING new_customer_data AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET city = source.city, segment = source.segment
WHEN NOT MATCHED THEN INSERT *
```

Without Delta Lake, achieving the same effect by hand would require: reading the whole old table, finding the non-matching rows, joining with the new data, writing EVERYTHING back from scratch -- a lot more code and a much bigger cost (rewriting the whole table instead of only the changed files).

<a id="sec5"></a>
## 5. Schema evolution

What happens when new data has an EXTRA column that wasn't in the original table? By
default, Delta Lake (like Parquet) raises a schema mismatch error -- but this can be
deliberately enabled via the `mergeSchema` option.

In [ ]:
from pyspark.sql import functions as F

customers_with_new_column = customers.withColumn("newsletter", F.lit(True))

# Without mergeSchema -- this would raise an AnalysisException (schema mismatch)
customers_with_new_column.write.format("delta").mode("append").option(
    "mergeSchema", "true"
).save("delta_data/customers_delta")

spark.read.format("delta").load("delta_data/customers_delta").printSchema()


> ⚠️ **mergeSchema is a deliberate decision, not an 'always on' setting**
>
> Turning on `mergeSchema` by default for ALL writes can be risky -- a typo in a column name in your source code (e.g. `custmer_id` instead of `customer_id`), instead of raising an error right away, will silently ADD an unwanted new column to the table. Turn on `mergeSchema` deliberately, only where you genuinely expect schema evolution (e.g. a new field added by the team producing the source data).

<a id="sec6"></a>
## 6. Summary and exercises

In this module we covered:
- why Delta Lake exists -- the limitations of plain Parquet (no ACID, no schema enforcement, no history, no update/delete),
- `_delta_log/` -- the JSON transaction log underlying every Delta table,
- time travel: `VERSION AS OF`/`versionAsOf`, `TIMESTAMP AS OF`, `.history()`, `RESTORE`,
- `MERGE INTO` -- an upsert (insert + update + optionally delete) in one statement,
- schema evolution via `mergeSchema`.

Remember: this module is fully ready to run in **Databricks Free Edition** -- there, all
the syntax above works with no extra configuration at all.

### 📝 Exercise 1: Writing and reading in Delta format

Write `products` in `delta` format to the folder `delta_data/products_delta`, then read it back and print `count()`.

<details>
<summary>Show solution</summary>

```python
products.write.format("delta").mode("overwrite").save("delta_data/products_delta")
loaded = spark.read.format("delta").load("delta_data/products_delta")
print(f"Row count: {loaded.count()}")
```

</details>

### 📝 Exercise 2: Reviewing the history

For the `products_delta` table from exercise 1, perform one more write (`mode("append")` with any subset of `products`), then print `.history()` -- how many versions does the table have now?

<details>
<summary>Show solution</summary>

```python
from delta.tables import DeltaTable

products.filter(products.product_id <= 10).write.format("delta").mode("append").save(
    "delta_data/products_delta"
)

table = DeltaTable.forPath(spark, "delta_data/products_delta")
table.history().select("version", "timestamp", "operation").show()
```

</details>

### 📝 Exercise 3: Reading an older version

Read VERSION 0 of the `products_delta` table (before the write in exercise 2) via `option("versionAsOf", 0)` and compare its `count()` to the latest version.

<details>
<summary>Show solution</summary>

```python
version_0 = spark.read.format("delta").option("versionAsOf", 0).load(
    "delta_data/products_delta"
)
latest = spark.read.format("delta").load("delta_data/products_delta")
print(f"Version 0: {version_0.count()}, latest: {latest.count()}")
```

</details>

### 📝 Exercise 4: A simple MERGE INTO

For the `products_delta` table, perform a `MERGE INTO` that raises `price` by 10% for the product with `product_id = 1` (if it exists), and if `product_id = 999` does not exist -- inserts it as a new product.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql import Row

new_data = spark.createDataFrame([
    Row(product_id=1, category="Electronics", price=None),
    Row(product_id=999, category="Electronics", price=99.99),
])

table.alias("target").merge(
    new_data.alias("source"), "target.product_id = source.product_id"
).whenMatchedUpdate(set={"price": "target.price * 1.1"}).whenNotMatchedInsertAll().execute()

table.toDF().filter("product_id IN (1, 999)").show()
```

</details>

> 🔥 **Challenge: a full Delta table lifecycle**
>
> Build a full scenario: (1) write `orders` as a Delta table; (2) perform a `MERGE INTO` that changes the status of ALL orders for `customer_id = 1` to `'Cancelled'`; (3) check `.history()` -- there should be TWO versions (the write + the MERGE); (4) read VERSION 0 and compare customer 1's order statuses before and after the MERGE.

<details>
<summary>Show solution</summary>

```python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

orders.write.format("delta").mode("overwrite").save("delta_data/orders_delta")
orders_table = DeltaTable.forPath(spark, "delta_data/orders_delta")

customer_1_new = orders.filter(orders.customer_id == 1).withColumn(
    "status", F.lit("Cancelled")
)

orders_table.alias("target").merge(
    customer_1_new.alias("source"), "target.order_id = source.order_id"
).whenMatchedUpdate(set={"status": "source.status"}).execute()

orders_table.history().select("version", "operation").show()

before = spark.read.format("delta").option("versionAsOf", 0).load(
    "delta_data/orders_delta"
).filter("customer_id = 1")
after = orders_table.toDF().filter("customer_id = 1")

print("Statuses before MERGE:")
before.groupBy("status").count().show()
print("Statuses after MERGE:")
after.groupBy("status").count().show()
```

This is exactly the pattern used in real data pipelines: 'mark a customer's cancelled orders' without rewriting the whole table from scratch, with a full history of changes available for auditing.

</details>

## What's next?

In **Module 11** we'll build a full picture of the **lakehouse architecture**: the
bronze/silver/gold layers (the Medallion Architecture), the differences between a data
lake, a data warehouse, and a lakehouse, and the basics of Unity Catalog.